## PATHS

In [ ]:
py_ds_path = "" # Path to generation dataset for python
js_ds_path = "" # Path to generation dataset for js
output_dir = "" # Ooutput dir for SFTConfig
save_path = "" # Save the model
merge_model_path = "" # Save the models with their weights merged

In [ ]:
!pip install transformers
!pip install datasets
!pip install peft
!pip install bitsandbytes
!pip install accelerate
!pip install trl
!pip install triton
!pip install sentencepiece
!pip install datasets
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.2 MB/s eta 0:00:00


In [ ]:
import torch
torch.cuda.is_available()

True

In [ ]:
from datasets import load_dataset, concatenate_datasets
import os

if os.path.exists(py_ds_path) and os.path.exists(js_ds_path):
    print("Loading Dataset")
    py_ds = load_dataset('json',data_files=py_ds_path, split='train')
    js_ds = load_dataset('json',data_files=js_ds_path, split='train')
    ds = concatenate_datasets([py_ds,js_ds])
    ds = ds.shuffle(seed=42)
    print("Concatenated and shuffled datset")
else:
    print("Could not load dataset")

Loading Dataset


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Concatenated and shuffled datset


In [ ]:
def format_prompt(example):
    prompt = f"<|im_start|>system\nYou are a helpful programming assistant.<|im_end|>\n"
    prompt += f"<|im_start|>user\n{example['description']}<|im_end|>\n"
    prompt += f"<|im_start|>assistant\n{example['code']}<|im_end|>"
    return prompt


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

model_id = "Qwen/CodeQwen1.5-7B"

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)

print("Loading qwen")
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto", trust_remote_code=True)
print("Qwen loaded successfully")
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(r=16, lora_alpha=32, target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_up_proj", "down_proj"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM")
model = get_peft_model(model, lora_config)
print("Lora added")

model.print_trainable_parameters()

In [ ]:
ex = format_prompt(ds[0])
ex['text']

'<|im_start|>system\nYou are a helpful programming assistant.<|im_end|>\n<|im_start|>user\nIf the given dtype is a DatetimeTZDtype, extract the implied\r\n    tzinfo object from it and check that it does not conflict with the given\r\n    tz.\r\n\r\n    Parameters\r\n    ----------\r\n    dtype : dtype, str\r\n    tz : None, tzinfo\r\n    explicit_tz_none : bool, default False\r\n        Whether tz=None was passed explicitly, as opposed to lib.no_default.\r\n\r\n    Returns\r\n    -------\r\n    tz : consensus tzinfo\r\n\r\n    Raises\r\n    ------\r\n    ValueError : on tzinfo mismatch<|im_end|>\n<|im_start|>assistant\ndef _validate_tz_from_dtype(\r\n    dtype, tz: tzinfo | None, explicit_tz_none: bool = False\r\n) -> tzinfo | None:\r\n    \r\n    if dtype is not None:\r\n        if isinstance(dtype, str):\r\n            try:\r\n                dtype = DatetimeTZDtype.construct_from_string(dtype)\r\n            except TypeError:\r\n                # Things like `datetime64[ns]`, which

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir=output_dir,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    max_steps=150,
    optim="paged_adamw_8bit",
    save_steps=50,
    save_total_limit=2,
    fp16=True,
    packing=True,
)

trainer = SFTTrainer(
    model=model,
    formatting_func=format_prompt,
    train_dataset=ds,
    peft_config=lora_config,
    processing_class=tokenizer,
    args=training_args,
)

print("Trainer ready")


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:348: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:196: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Applying formatting function to train dataset:   0%|          | 0/37851 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/37851 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/37851 [00:00<?, ? examples/s]

Packing train dataset:   0%|          | 0/37851 [00:00<?, ? examples/s]

Trainer ready


In [ ]:
print("Start fine tuning")
trainer.train()

In [ ]:
trainer.save_model(save_path)

In [ ]:
import gc
del model
gc.collect()
torch.cuda.empty_cache()
gc.collect()

0

In [ ]:
import torch
torch.cuda.is_available()
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
model_id = "Qwen/CodeQwen1.5-7B"

In [ ]:
from peft import PeftModel
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16,bnb_4bit_use_double_quant=True)
base_model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config = bnb_config, device_map = "auto", trust_remote_code = True)
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
model = PeftModel.from_pretrained(base_model, save_path)
model = model.merge_and_unload()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/702 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/387 [00:00<?, ?it/s]

...

The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


tokenizer_config.json:   0%|          | 0.00/619 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/1.42M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:397: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


In [ ]:
model.generation_config.top_p = None
model.save_pretrained(merge_model_path)
tokenizer.save_pretrained(merge_model_path)

In [ ]:
description = "Create a simple Python FastAPI main function."

prompt = f"<|im_start|>system\nYou are a helpful programming assistant.<|im_end|>\n"
prompt += f"<|im_start|>user\n{description}<|im_end|>\n"
prompt += f"<|im_start|>assistant\n"

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
inputs.pop("token_type_ids",None)
input_token_length = inputs["input_ids"].shape[1]

print("Generating response")
outputs = model.generate(**inputs, max_new_tokens=100, eos_token_id=tokenizer.eos_token_id)

response_tokens = outputs[0][input_token_length:]
response = tokenizer.decode(response_tokens, skip_special_tokens=True)

print("--- Model Response ---")
print(response)

Generating response
--- Model Response ---
Sure, here's an example of a simple Python FastAPI main function:

```python
from fastapi import FastAPI

app = FastAPI()

@app.get("/")
def read_root():
    return {"Hello": "World"}

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)
```


In [ ]:
!git clone https://github.com/bigcode-project/bigcode-evaluation-harness
%cd bigcode-evaluation-harness
!pip install -e .
%cd ..


Cloning into 'lm-evaluation-harness'...
remote: Enumerating objects: 62984, done.
remote: Counting objects: 100% (45/45), done.
remote: Compressing objects: 100% (27/27), done.
remote: Total 62984 (delta 29), reused 18 (delta 18), pack-reused 62939 (from 2)
Receiving objects: 100% (62984/62984), 34.34 MiB | 16.42 MiB/s, done.
Resolving deltas: 100% (43498/43498), done.
/content/bigcode-evaluation-harness/lm-evaluation-harness
Obtaining file:///content/bigcode-evaluation-harness/lm-evaluation-harness
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 11.1 MB/s eta 0:00:00
  Bu

In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()
!nvidia-smi

Sat Mar 21 11:43:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   73C    P0             30W /   70W |     239MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# REPLACE PATHS HERE

In [ ]:
%cd /content/bigcode-evaluation-harness
!accelerate launch main.py \
  --model """ REPLACE WITH MERGED_MODEL_PATH USING / AS SEPARATORS""" \
  --tasks humanevalsynthesize-python,humanevalsynthesize-js \
  --do_sample False \
  --n_samples 1 \
  --batch_size 1 \
  --precision fp16 \
  --allow_code_execution \
  --save_generations \
  --prompt octocoder \
  --max_length_generation 2048 \
  --metric_output_path """ REPLACE WITH OUTPUT PATH """

/content/bigcode-evaluation-harness
The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_processes` was set to a value of `1`
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
Selected Tasks: ['humanevalsynthesize-python', 'humanevalsynthesize-js']
Loading model in fp16
Loading weights: 100% 387/387 [00:19<00:00, 19.96it/s, Materializing param=model.norm.weight]
number of problems for this task is 164
100% 164/164 [27:48<00:00, 10.17s/it]
generations were saved at generations_humanevalsynthesize-python.json
Evaluating generations...
number of problems for this task is 164
100% 164/164 [34:09<00:00, 12.49s/it]
generations were saved at generations_humanevalsynthesize-js.json
Evaluating generations...
{
  "humanevalsynthesize-python": {
    "pass@1"